# 04 · Словарь значений датчиков и статусы неисправностей

**Цель** — исследовать `значение_датчика`: числовые показания (телеметрия) vs текстовые статусы, выделить статусы-маркеры неисправностей («Неисправен», «Обесточен», «Затоплен» и т.п.). Это кандидаты в **целевую переменную** для прогноза отказов.

In [ ]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
if _HERE.name == "notebooks":
    RESEARCH = _HERE.parent
else:
    RESEARCH = _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import pandas as pd
import data_utils as du

RECOMPUTE = True
print("dataset:", du.find_dataset_dir())

# 4.1 · Словарь значений (артефакт `value_counts_all_years.csv`)

Полный пересчёт по всем годам (~минуты). При `RECOMPUTE=False` читаем готовый CSV.

In [ ]:
out_csv = du.find_dataset_dir() / "value_counts_all_years.csv"
if RECOMPUTE or not out_csv.exists():
    all_vals = {}
    for fp in du.journal_files():
        for ch in du.read_chunks(fp, cols=["значение_датчика", "ид_канала_данных"]):
            for v, c in ch["значение_датчика"].fillna("(пусто)").value_counts().items():
                all_vals[v] = all_vals.get(v, 0) + int(c)
    vc = pd.Series(all_vals).sort_values(ascending=False)
    vc.rename("счет").to_csv(out_csv, encoding="utf-8-sig")
else:
    vc = pd.read_csv(out_csv, index_col=0).iloc[:, 0]
    print("Прочитано из готового артефакта:", out_csv)

print("Уникальных значений:", len(vc))
print(vc.head(25).to_string())

# 4.2 · Текстовые статусы

Числовые значения — показания датчиков (телеметрия). Текстовые — статусы: они и есть «семантическая» часть. Отделяем и сохраняем в `text_status_counts_all_years.csv`.

In [ ]:
text = vc[~vc.index.map(du.is_numeric_value)]
text.rename("счет").to_csv(du.find_dataset_dir() / "text_status_counts_all_years.csv",
                           encoding="utf-8-sig")
print("Текстовых статусов:", len(text))
print(text.head(30).to_string())

# 4.3 · Статусы-маркеры неисправностей

Ключевые кандидаты в целевую переменную (по ключевым словам). **Не путать:** флаг `тревожное` — «сигнал тревоги», а статусы — «техническое состояние датчика».

In [ ]:
fault_kw = ["исправ", "авар", "неис", "обест", "ошибк", "откл", "демонт",
            "нет связи", "задым", "пожар", "затоп", "отказ", "взлом"]
fault = text[text.index.map(lambda v: any(kw in v.lower() for kw in fault_kw))]
print("Статусы-маркеры неисправностей (кол-во событий за все годы):")
print(fault.to_string())

# 4.4 · Топ-статусы по ключевым типам датчиков

Проверим, какие статусы характерны для дыма, газа, температуры, насосов, дверей — это определяет **правила разметки цели**.

In [ ]:
ref_ch = du.load_ref_channels()[["ид_канала_данных", "тип_датчика"]]
targets = ["Датчик дыма", "Датчик температуры", "Газовый датчик",
           "Состояние насоса", "КД Дверь", "Датчик движения"]
files = du.journal_files()
for t in targets:
    stats = {}
    for fp in files:
        for ch in du.read_chunks(fp, cols=["ид_канала_данных", "значение_датчика"]):
            m = ch.merge(ref_ch, on="ид_канала_данных", how="left")
            m = m[m["тип_датчика"] == t]
            for v, c in m["значение_датчика"].fillna("(пусто)").value_counts().items():
                if du.is_numeric_value(v):
                    continue
                stats[v] = stats.get(v, 0) + int(c)
    top = pd.Series(stats).sort_values(ascending=False)
    print(f"\n-- {t} (топ-8 текстовых статусов) --")
    print(top.head(8).to_string())

# 4.5 · Комментарии

- **«Неисправен» — 1.5 млн событий**, из них ≈1.17 млн приходится на датчики дыма: отличный, массово размеченный целевой маркер для **«Отказа датчика»**.
- **«Обесточен» — 1.29 млн**: признак/цель для диагностики питания (ИБП, насосы).
- **«Затоплен» — 64k**: для рисков подтопления.
- Для прогноза отказа используем **статусы**; для прогноза рисков (пожар/доступ) — флаг тревоги + типы датчиков.
- **Разметка цели** «отказ в следующие 24 ч»: наличие статуса «Неисправен/Обесточен/Отключено устройство» у канала в окне +24 ч.